# Threshold tuning only (no retraining)

Loads the **5-epoch** TinyBERT from Drive (`tinybert_from_official_L256_T4_e5`), searches per-label thresholds on a **train-carved val split**, then scores **once** on public test with full competition metrics.

**Keep rule:** accept tuned thresholds only if `submission_score >= 0.8555` (e5 @0.5 public score). Otherwise keep 0.5/0.5.

Writes e5-specific JSONs (does **not** overwrite e3 threshold files).

GPU recommended for full-score timing/FLOPs; threshold search itself is fine on CPU.

In [1]:
!pip -q install "transformers>=4.36.0" "datasets>=2.16.0" "pandas" "numpy" "tqdm" "scikit-learn" "accelerate" "safetensors"

import json, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/NLBSE27_DistilVulSATD')
RUN_NAME = 'tinybert_from_official_L256_T4_e5'  # matches 03_distill_student-epoch-5
BEST_DIR = DRIVE_ROOT / 'checkpoints' / 'student_runs' / RUN_NAME / 'best_model'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_SCORE = 0.8555  # e5 @0.5 public score; accept tuned only if >= this
MAX_LENGTH = 256
SEED = 42
VAL_FRACTION = 0.05

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
print('BEST_DIR exists?', BEST_DIR.exists())
assert BEST_DIR.exists(), 'Missing TinyBERT checkpoint on Drive'

Mounted at /content/drive
device = cuda
BEST_DIR exists? True


In [2]:
def parse_label(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return [int(v[0]), int(v[1])]
    nums = re.findall(r'-?\d+', str(v))
    return [int(nums[0]), int(nums[1])]


def f1_binary(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return (2 * p * r / (p + r)) if (p + r) else 0.0, p, r


def f1_table(preds, labels):
    rows = []
    for i, name in enumerate(['Vulnerability', 'MAT']):
        f1, p, r = f1_binary(labels[:, i], preds[:, i])
        rows.append({'cat': name, 'precision': p, 'recall': r, 'f1': f1})
    df = pd.DataFrame(rows)
    return df, float(df.f1.mean())


class FnDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = list(texts)
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        return {k: v.squeeze(0) for k, v in enc.items()}


@torch.no_grad()
def predict_proba(model, tokenizer, texts, batch_size=64):
    model.eval()
    loader = DataLoader(FnDataset(texts, tokenizer, MAX_LENGTH), batch_size=batch_size)
    probs = []
    for batch in tqdm(loader, desc='predict', leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(probs, 0)


def apply_thresholds(probs, thr_vuln, thr_satd):
    out = np.zeros_like(probs, dtype=np.int32)
    out[:, 0] = (probs[:, 0] > thr_vuln).astype(np.int32)
    out[:, 1] = (probs[:, 1] > thr_satd).astype(np.int32)
    return out


print('helpers ready')

helpers ready


In [3]:
# Data: tune on carved val from TRAIN only; report once on public test
ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df['label_list'] = train_df['Label'].apply(parse_label)
test_df['label_list'] = test_df['Label'].apply(parse_label)
train_df['vuln'] = train_df['label_list'].apply(lambda x: x[0])
train_df['satd'] = train_df['label_list'].apply(lambda x: x[1])

joint = train_df['vuln'].astype(str) + '_' + train_df['satd'].astype(str)
_, val_df = train_test_split(
    train_df, test_size=VAL_FRACTION, random_state=SEED, stratify=joint
)
val_df = val_df.reset_index(drop=True)

val_texts = val_df['Function'].tolist()
test_texts = test_df['Function'].tolist()
val_labels = np.stack(val_df['label_list'].to_numpy())
test_labels = np.stack(test_df['label_list'].to_numpy())
print('val', len(val_df), 'test', len(test_df))

README.md:   0%|          | 0.00/530 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/214586 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30655 [00:00<?, ? examples/s]

val 10730 test 30655


In [4]:
tokenizer = AutoTokenizer.from_pretrained(BEST_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR)
model.to(device)

val_probs = predict_proba(model, tokenizer, val_texts, batch_size=64)
test_probs = predict_proba(model, tokenizer, test_texts, batch_size=64)

# baseline thresholds 0.5/0.5 on val
base_pred = apply_thresholds(val_probs, 0.5, 0.5)
base_df, base_f1 = f1_table(base_pred, val_labels)
print('VAL @0.5/0.5')
print(base_df.to_string(index=False))
print('val avg F1', base_f1)

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

VAL @0.5/0.5
          cat  precision   recall       f1
Vulnerability   0.842381 0.776293 0.807988
          MAT   0.850394 0.610169 0.710526
val avg F1 0.7592572407349736


In [5]:
# Grid search thresholds on VAL only (maximize avg F1; break ties by MAT F1 then vuln F1)
grid = [round(x, 2) for x in np.arange(0.05, 0.96, 0.05)]
best = {
    'thr_vuln': 0.5,
    'thr_satd': 0.5,
    'avg_f1': base_f1,
    'vuln_f1': float(base_df.loc[base_df.cat=='Vulnerability', 'f1'].iloc[0]),
    'satd_f1': float(base_df.loc[base_df.cat=='MAT', 'f1'].iloc[0]),
}

rows = []
for tv in tqdm(grid, desc='thr vuln'):
    for ts in grid:
        pred = apply_thresholds(val_probs, tv, ts)
        vuln_f1, _, _ = f1_binary(val_labels[:, 0], pred[:, 0])
        satd_f1, _, _ = f1_binary(val_labels[:, 1], pred[:, 1])
        avg = 0.5 * (vuln_f1 + satd_f1)
        rows.append({'thr_vuln': tv, 'thr_satd': ts, 'vuln_f1': vuln_f1, 'satd_f1': satd_f1, 'avg_f1': avg})
        key = (avg, satd_f1, vuln_f1)
        best_key = (best['avg_f1'], best['satd_f1'], best['vuln_f1'])
        if key > best_key:
            best = {
                'thr_vuln': tv,
                'thr_satd': ts,
                'avg_f1': avg,
                'vuln_f1': vuln_f1,
                'satd_f1': satd_f1,
            }

print('BEST VAL THRESHOLDS:', best)
pd.DataFrame(rows).sort_values(['avg_f1', 'satd_f1'], ascending=False).head(10)

thr vuln: 100%|██████████| 19/19 [00:00<00:00, 144.61it/s]

BEST VAL THRESHOLDS: {'thr_vuln': np.float64(0.2), 'thr_satd': np.float64(0.55), 'avg_f1': 0.7620698372918058, 'vuln_f1': 0.8112683874548986, 'satd_f1': 0.712871287128713}


,thr_vuln,thr_satd,vuln_f1,satd_f1,avg_f1
67,0.20,0.55,0.811268,0.712871,0.762070
105,0.30,0.55,0.811157,0.712871,0.762014
48,0.15,0.55,0.810922,0.712871,0.761896
86,0.25,0.55,0.810241,0.712871,0.761556
124,0.35,0.55,0.809777,0.712871,0.761324
29,0.10,0.55,0.809441,0.712871,0.761156
143,0.40,0.55,0.809067,0.712871,0.760969
65,0.20,0.45,0.811268,0.710526,0.760897
66,0.20,0.50,0.811268,0.710526,0.760897
103,0.30,0.45,0.811157,0.710526,0.760842


In [6]:
# Public-test comparison: 0.5/0.5 vs tuned thresholds
def score_public(probs, thr_vuln, thr_satd, measure_efficiency=True):
    preds = apply_thresholds(probs, thr_vuln, thr_satd)
    sdf, avg_f1 = f1_table(preds, test_labels)

    # Efficiency uses model forward cost (independent of threshold), match competition protocol
    if measure_efficiency and device.type == 'cuda':
        repetitions, batch_size = 10, 600
        tokenized = tokenizer(
            test_texts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH,
        )
        loader = DataLoader(
            TensorDataset(tokenized['input_ids'], tokenized['attention_mask']),
            batch_size=batch_size,
        )
        total_time = 0.0
        total_flops = 0.0
        pbar = tqdm(total=len(test_texts), desc='full score')
        model.eval()
        with torch.no_grad():
            for batch in loader:
                input_ids, attention_mask = [x.to(device, non_blocking=True) for x in batch]
                begin = time.time()
                with torch.amp.autocast('cuda'):
                    with torch.profiler.profile(with_flops=True) as prof:
                        for _ in range(repetitions):
                            _ = model(input_ids=input_ids, attention_mask=attention_mask)
                    total_flops += sum(k.flops for k in prof.key_averages()) / 1e9
                total_time += time.time() - begin
                pbar.update(input_ids.size(0))
                del input_ids, attention_mask
                torch.cuda.empty_cache()
        pbar.close()
        avg_runtime = total_time / repetitions
        avg_flops = total_flops / repetitions
    else:
        # Fallback if no GPU: reuse previous clean-run efficiency numbers
        avg_runtime = 48.54978859424591
        avg_flops = 81535.36562038395
        print('NOTE: no CUDA — reusing prior runtime/FLOPs from clean run for score formula')

    submission = (
        0.6 * avg_f1
        + 0.2 * max((1000 - avg_runtime) / 1000, 0)
        + 0.2 * max((3_000_000 - avg_flops) / 3_000_000, 0)
    )
    return {
        'thr_vuln': thr_vuln,
        'thr_satd': thr_satd,
        'avg_f1': avg_f1,
        'avg_runtime': avg_runtime,
        'avg_flops': avg_flops,
        'submission_score': float(submission),
        'per_class': sdf.to_dict(orient='records'),
        'score_df': sdf,
    }

base_pub = score_public(test_probs, 0.5, 0.5, measure_efficiency=True)
tuned_pub = score_public(
    test_probs, best['thr_vuln'], best['thr_satd'], measure_efficiency=False
)  # reuse efficiency from first call
tuned_pub['avg_runtime'] = base_pub['avg_runtime']
tuned_pub['avg_flops'] = base_pub['avg_flops']
tuned_pub['submission_score'] = (
    0.6 * tuned_pub['avg_f1']
    + 0.2 * max((1000 - tuned_pub['avg_runtime']) / 1000, 0)
    + 0.2 * max((3_000_000 - tuned_pub['avg_flops']) / 3_000_000, 0)
)

print('PUBLIC @0.5/0.5')
print(base_pub['score_df'].to_string(index=False))
print('avg F1', base_pub['avg_f1'], 'score', round(base_pub['submission_score'], 4))

print('\nPUBLIC @tuned', best['thr_vuln'], best['thr_satd'])
print(tuned_pub['score_df'].to_string(index=False))
print('avg F1', tuned_pub['avg_f1'], 'score', round(tuned_pub['submission_score'], 4))

full score: 100%|██████████| 30655/30655 [07:12<00:00, 70.85it/s]


NOTE: no CUDA — reusing prior runtime/FLOPs from clean run for score formula
PUBLIC @0.5/0.5
          cat  precision   recall       f1
Vulnerability   0.849506 0.787306 0.817224
          MAT   0.945455 0.617822 0.747305
avg F1 0.7822648383530337 score 0.8553

PUBLIC @tuned 0.2 0.55
          cat  precision   recall       f1
Vulnerability   0.802830 0.840927 0.821437
          MAT   0.945455 0.617822 0.747305
avg F1 0.7843713185843462 score 0.8566


In [7]:
# Accept/reject rule
accept = tuned_pub['submission_score'] >= CURRENT_SCORE
final_thr = {
    'thr_vuln': best['thr_vuln'] if accept else 0.5,
    'thr_satd': best['thr_satd'] if accept else 0.5,
}
final_score = tuned_pub if accept else base_pub

payload = {
    'run_name': RUN_NAME,
    'current_score_reference': CURRENT_SCORE,
    'val_best_thresholds': best,
    'public_baseline_0_5': {
        k: v for k, v in base_pub.items() if k != 'score_df'
    },
    'public_tuned': {
        k: v for k, v in tuned_pub.items() if k != 'score_df'
    },
    'accepted': bool(accept),
    'final_thresholds': final_thr,
    'final_submission_score': float(final_score['submission_score']),
    'final_avg_f1': float(final_score['avg_f1']),
    'checkpoint': str(BEST_DIR),
    'rule': f'Accept tuned thresholds only if public submission_score >= {CURRENT_SCORE}',
}

# e5-specific filenames — leave e3 threshold_tune_result.json / final_thresholds.json untouched
out_json = RESULTS_DIR / f'threshold_tune_result_{RUN_NAME}.json'
thr_json = RESULTS_DIR / f'final_thresholds_{RUN_NAME}.json'
with open(out_json, 'w') as f:
    json.dump(payload, f, indent=2)
with open(thr_json, 'w') as f:
    json.dump({
        **final_thr,
        'run_name': RUN_NAME,
        'accepted_from_tuning': bool(accept),
        'submission_score': float(final_score['submission_score']),
        'avg_f1': float(final_score['avg_f1']),
    }, f, indent=2)

print('ACCEPTED TUNED?' , accept)
print('FINAL thresholds:', final_thr)
print('FINAL submission_score:', round(final_score['submission_score'], 4))
print('FINAL avg F1:', final_score['avg_f1'])
print('SAVED', out_json)
print('SAVED', thr_json)
if accept:
    print('Use these thresholds at inference instead of 0.5/0.5')
else:
    print(f'Keep 0.5/0.5 — tuning did not beat {CURRENT_SCORE} on public competition score')

ACCEPTED TUNED? True
FINAL thresholds: {'thr_vuln': np.float64(0.2), 'thr_satd': np.float64(0.55)}
FINAL submission_score: 0.8566
FINAL avg F1: 0.7843713185843462
SAVED /content/drive/MyDrive/NLBSE27_DistilVulSATD/results/threshold_tune_result_tinybert_from_official_L256_T4_e5.json
SAVED /content/drive/MyDrive/NLBSE27_DistilVulSATD/results/final_thresholds_tinybert_from_official_L256_T4_e5.json
Use these thresholds at inference instead of 0.5/0.5
